### 33_difhistoryplayer_2

Script: build_mnm_pipeline.py
* denna notebook [33_difhistoryplayer_2.ipynb](https://github.com/salgo60/ifkdb/blob/main/Notebook/33_difhistoryplayer_2.ipynb)
   * [mnm_update_difplayers.csv](https://github.com/salgo60/ifkdb/blob/main/Notebook/mnm_update_difplayers.csv)
   * [quickstatements_difplayers.txt](https://github.com/salgo60/ifkdb/blob/main/Notebook/quickstatements_difplayers.txt)
 
Vibekodad
* ✔ parallell scraping
* ✔ progressbar
* ✔ undviker dubbel P54
* ✔ skapar både Mix'n'Match + QuickStatements
* ✔ fungerar direkt på ditt dataset (~1100 poster)

In [1]:
from datetime import datetime
import platform
import sys
import os

# Start timer
start_time = datetime.now()
print(f"Start time: {start_time:%Y-%m-%d %H:%M:%S}")


Start time: 2026-03-07 09:10:52


In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import re

tqdm.pandas()

# --- load data ---
df = pd.read_json("difplayers.json")

# --- scrape infobox ---
def scrape(url):
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        box = soup.select_one("section.wp-block-uagb-section")
        return box.get_text("\n", strip=True) if box else ""
    except:
        return ""

with ThreadPoolExecutor(max_workers=12) as ex:
    df["infobox"] = list(tqdm(ex.map(scrape, df["external_url"]), total=len(df)))

# --- extract birth year ---
df["born_year"] = df["infobox"].str.extract(r"Född:\s*\d+\s+\w+\s+(\d{4})")

# --- extract position ---
df["position"] = df["infobox"].str.extract(r"Position:\s*([^\n]+)")

pos_map = {
    "målvakt": "goalkeeper",
    "back": "defender",
    "mittfältare": "midfielder",
    "anfallare": "forward"
}

df["position"] = df["position"].replace(pos_map)

# --- extract club years (last two years in infobox) ---
df["years"] = df["infobox"].str.findall(r"\b(?:19|20)\d{2}\b")

def club_range(x):
    if len(x) >= 2:
        return f"{x[-2]}-{x[-1]}"
    return None

df["club_years"] = df["years"].apply(club_range)

# --- create Mix'n'Match description ---
df["desc"] = "football " + df["position"].fillna("player")
df["desc"] += " (born " + df["born_year"].fillna("") + ")"

df.loc[df["club_years"].notna(), "desc"] += ", Djurgårdens IF player " + df["club_years"]

df["desc"] = df["desc"].str.replace(r"\s+", " ", regex=True)

# --- Mix'n'Match export ---
mnm = df[["external_id", "name", "desc"]].rename(columns={"external_id": "id"})
mnm["desc"] = mnm["desc"].str.slice(0,150)
mnmfile = "mnm_update_difplayers.csv"
mnm.to_csv(mnmfile, index=False)

# --- create QuickStatements ---
qs_rows = []

for _, r in df.iterrows():
    if pd.notna(r["q"]):

        if pd.notna(r["born_year"]):
            qs_rows.append(f'{r["q"]}\tP569\t+{r["born_year"]}-00-00T00:00:00Z/9')

        if pd.notna(r["club_years"]):
            start,end = r["club_years"].split("-")
            qs_rows.append(f'{r["q"]}\tP54\tQ163131\tP580\t+{start}-00-00T00:00:00Z/9\tP582\t+{end}-00-00T00:00:00Z/9')

qs = pd.DataFrame(qs_rows)
qsfile = "quickstatements_difplayers.txt"
qs.to_csv(qsfile, index=False, header=False)

print("Files created:")
print(mnmfile)
print(qsfile)

100%|███████████████████████████████████████| 1105/1105 [02:51<00:00,  6.43it/s]

Files created:
mnm_update_difplayers.csv
quickstatements_difplayers.txt


In [3]:
# End timer
end_time = datetime.now()
duration = end_time - start_time

print("\n--- Execution Summary ---")
print(f"End time:      {end_time:%Y-%m-%d %H:%M:%S}")
print(f"Duration:      {duration}")
print(f"Total seconds: {duration.total_seconds():.2f}")
print(f"Python ver:    {sys.version.split()[0]}")
print(f"Platform:      {platform.system()} {platform.release()}")
print(f"Process ID:    {os.getpid()}")


--- Execution Summary ---
End time:      2026-03-07 09:13:44
Duration:      0:02:52.289087
Total seconds: 172.29
Python ver:    3.12.2
Platform:      Darwin 25.3.0
Process ID:    12500
